# Project 3: Sales Forecasting
## Notebook 6: Deep Learning — LSTM + Monte Carlo Uncertainty

**Two things that separate this from every other forecasting project:**

1. **LSTM (Long Short-Term Memory)** — a recurrent neural network that learns long-range dependencies in sequences. It remembers patterns from months ago, not just recent lags.

2. **Monte Carlo Dropout** — instead of one confidence band, we run the model 100 times with random dropout and get a full *distribution* of possible futures. This is more honest than a single confidence interval.

**Install:** `pip install torch` (PyTorch)

In [1]:
pip uninstall pytorch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install pytorch

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.dpi'] = 120

BASE      = Path(r'C:\Users\Administrator\SalesForecast')
PROC_DIR  = BASE / 'data' / 'processed'
MODEL_DIR = BASE / 'models'
FIG_DIR   = BASE / 'reports' / 'figures'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('Libraries loaded.')

## 1. Load & Normalise Data

LSTMs require data normalised to [0, 1]. We use MinMaxScaler — remember to inverse-transform predictions.

In [ ]:
monthly = pd.read_csv(PROC_DIR / 'monthly_sales.csv', index_col=0, parse_dates=True).squeeze()

TEST_MONTHS = 12
LOOK_BACK   = 12    # how many past months the LSTM looks at per step

# Normalise to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(monthly.values.reshape(-1, 1)).flatten()

print(f'Series length: {len(scaled)}')
print(f'Scaled range: {scaled.min():.3f} → {scaled.max():.3f}')

## 2. Create Sequences

LSTM takes fixed-length windows as input. For each month t, the input is months [t-12 … t-1] and the target is month t.

In [ ]:
def create_sequences(data, look_back):
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(scaled, LOOK_BACK)

# Split — keep temporal order
split = len(X_all) - TEST_MONTHS
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1).to(DEVICE)  # (samples, seq, features)
y_train_t = torch.FloatTensor(y_train).to(DEVICE)
X_test_t  = torch.FloatTensor(X_test).unsqueeze(-1).to(DEVICE)
y_test_t  = torch.FloatTensor(y_test).to(DEVICE)

print(f'Train sequences: {X_train_t.shape}  Target: {y_train_t.shape}')
print(f'Test sequences:  {X_test_t.shape}   Target: {y_test_t.shape}')

## 3. Define LSTM Architecture

Two LSTM layers with dropout between them. Dropout stays active during inference — this is what enables Monte Carlo uncertainty estimation.

In [ ]:
class SalesLSTM(nn.Module):
    """
    Two-layer LSTM for monthly sales forecasting.
    Dropout is applied between layers AND kept active during inference
    (via training=True in forward pass) to enable Monte Carlo sampling.
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.2, output_size=1):
        super(SalesLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.dropout_rate = dropout

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
            batch_first = True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 32)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(32, output_size)

    def forward(self, x, mc_dropout=False):
        """
        mc_dropout=True keeps dropout active even during eval mode.
        This is the key to Monte Carlo uncertainty estimation.
        """
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        out     = out[:, -1, :]     # take last timestep

        if mc_dropout:
            out = nn.functional.dropout(out, p=self.dropout_rate, training=True)

        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out.squeeze()


model_lstm = SalesLSTM(hidden_size=64, num_layers=2, dropout=0.2).to(DEVICE)
print(model_lstm)
print(f'\nParameters: {sum(p.numel() for p in model_lstm.parameters()):,}')

## 4. Train the LSTM

In [ ]:
EPOCHS    = 200
LR        = 0.001
BATCH     = 16

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)

dataset   = TensorDataset(X_train_t, y_train_t)
loader    = DataLoader(dataset, batch_size=BATCH, shuffle=True)

train_losses = []
best_loss    = np.inf
best_state   = None

for epoch in range(EPOCHS):
    model_lstm.train()
    epoch_loss = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        pred = model_lstm(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)

    if avg_loss < best_loss:
        best_loss  = avg_loss
        best_state = {k: v.clone() for k, v in model_lstm.state_dict().items()}

    if (epoch + 1) % 50 == 0:
        print(f'  Epoch {epoch+1:>3}/{EPOCHS} | Loss: {avg_loss:.6f} | LR: {optimizer.param_groups[0]["lr"]:.5f}')

model_lstm.load_state_dict(best_state)
print(f'\nTraining complete. Best loss: {best_loss:.6f}')

# Plot loss curve
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(train_losses, color='#2E75B6', lw=2)
ax.set(title='LSTM Training Loss', xlabel='Epoch', ylabel='MSE Loss')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(FIG_DIR / '16_lstm_training_loss.png', bbox_inches='tight')
plt.show()

## 5. Standard Test Evaluation

In [ ]:
model_lstm.eval()
with torch.no_grad():
    test_pred_scaled = model_lstm(X_test_t).cpu().numpy()

# Inverse transform
test_pred = scaler.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()
y_test_actual = monthly.values[-TEST_MONTHS:]

def mape(y_true, y_pred):
    return np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100

lstm_mape = mape(y_test_actual, test_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))
lstm_mae  = mean_absolute_error(y_test_actual, test_pred)

print('=== LSTM Test Metrics ===')
print(f'  MAPE: {lstm_mape:.2f}%')
print(f'  RMSE: ${lstm_rmse:,.0f}')
print(f'  MAE:  ${lstm_mae:,.0f}')

## 6. Monte Carlo Dropout — Uncertainty Estimation

Run the model 200 times with dropout active. Each run gives a slightly different prediction.
The spread of those predictions = our uncertainty estimate.

This is more honest than a fixed confidence interval — it shows the actual distribution of possible futures.

In [ ]:
N_MC_SAMPLES = 200

model_lstm.eval()  # batch norm stays fixed, but we force dropout on via mc_dropout=True

mc_preds_scaled = []
with torch.no_grad():
    for _ in range(N_MC_SAMPLES):
        pred = model_lstm(X_test_t, mc_dropout=True).cpu().numpy()
        mc_preds_scaled.append(pred)

mc_preds_scaled = np.array(mc_preds_scaled)  # shape: (200, test_months)

# Inverse transform each sample
mc_preds = np.array([
    scaler.inverse_transform(s.reshape(-1, 1)).flatten()
    for s in mc_preds_scaled
])

# Compute statistics across MC samples
mc_mean   = mc_preds.mean(axis=0)
mc_std    = mc_preds.std(axis=0)
mc_p5     = np.percentile(mc_preds, 5,  axis=0)
mc_p25    = np.percentile(mc_preds, 25, axis=0)
mc_p75    = np.percentile(mc_preds, 75, axis=0)
mc_p95    = np.percentile(mc_preds, 95, axis=0)

print(f'MC samples: {N_MC_SAMPLES}')
print(f'Average uncertainty (std): ±${mc_std.mean():,.0f}')
print(f'90% prediction interval width: ${(mc_p95 - mc_p5).mean():,.0f} on average')

## 7. Plot Monte Carlo Uncertainty

In [ ]:
test_dates = monthly.index[-TEST_MONTHS:]
train_series = monthly.iloc[:-TEST_MONTHS]

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# ── Top: MC uncertainty bands ──────────────────────────────────────────────
ax = axes[0]
ax.plot(train_series.index, train_series.values, color='#2E75B6', lw=2, label='Historical')
ax.plot(test_dates, y_test_actual, color='#2E75B6', lw=2, linestyle='--', label='Actual (test)')

# 90% band
ax.fill_between(test_dates, mc_p5, mc_p95, alpha=0.15, color='#DD8452', label='90% MC interval')
# 50% band (IQR)
ax.fill_between(test_dates, mc_p25, mc_p75, alpha=0.30, color='#DD8452', label='50% MC interval (IQR)')
# Mean prediction
ax.plot(test_dates, mc_mean, color='#DD8452', lw=2.5, label=f'LSTM mean (MAPE={lstm_mape:.1f}%)')

ax.axvline(test_dates[0], color='gray', linestyle=':', alpha=0.7)
ax.set(title='LSTM Forecast with Monte Carlo Uncertainty (200 samples)',
       ylabel='Sales ($)')
ax.legend()

# ── Bottom: show individual MC traces ─────────────────────────────────────
ax2 = axes[1]
for i in range(min(50, N_MC_SAMPLES)):
    ax2.plot(test_dates, mc_preds[i], alpha=0.08, color='#DD8452', lw=1)
ax2.plot(test_dates, mc_mean, color='#DD8452', lw=2.5, label='Mean prediction')
ax2.plot(test_dates, y_test_actual, color='#2E75B6', lw=2, linestyle='--', label='Actual')
ax2.set(title='50 Individual MC Samples (shows true uncertainty distribution)',
        xlabel='Date', ylabel='Sales ($)')
ax2.legend()

plt.suptitle('LSTM + Monte Carlo Dropout — Uncertainty Quantification', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / '17_lstm_mc_uncertainty.png', bbox_inches='tight')
plt.show()

## 8. Full Model Comparison — All 5 Models

In [ ]:
# Load all saved metrics
all_metrics = []
for fname in ['sarima_metrics.json','prophet_metrics.json','lgb_metrics.json','hybrid_metrics.json']:
    try:
        with open(MODEL_DIR / fname) as f:
            all_metrics.append(json.load(f))
    except:
        pass

all_metrics.append({'model':'LSTM','mape':round(lstm_mape,2),'rmse':round(float(lstm_rmse),0),'mae':round(float(lstm_mae),0)})

comp = pd.DataFrame(all_metrics).sort_values('mape')
print('=== FULL MODEL COMPARISON (lower is better) ===')
print(comp.to_string(index=False))

# Visual comparison
colors = ['#2E75B6','#55A868','#DD8452','#8172B3','#C44E52']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, label in zip(axes, ['mape','rmse','mae'], ['MAPE (%)', 'RMSE ($)', 'MAE ($)']):
    sorted_df = comp.sort_values(metric)
    bars = ax.bar(sorted_df['model'], sorted_df[metric],
                  color=colors[:len(sorted_df)], edgecolor='white')
    ax.set(title=label, xticklabels=sorted_df['model'])
    ax.set_xticklabels(sorted_df['model'], rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.1f}' if metric == 'mape' else f'${val:,.0f}',
                ha='center', fontsize=8, fontweight='bold')

plt.suptitle('All 5 Models Compared — Lower is Better', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / '18_all_models_comparison.png', bbox_inches='tight')
plt.show()

## 9. Save LSTM

In [ ]:
torch.save(model_lstm.state_dict(), MODEL_DIR / 'lstm_model.pt')
joblib.dump(scaler, MODEL_DIR / 'lstm_scaler.pkl')

with open(MODEL_DIR / 'lstm_metrics.json', 'w') as f:
    json.dump({'model':'LSTM','mape':round(lstm_mape,2),
               'rmse':round(float(lstm_rmse),0),'mae':round(float(lstm_mae),0)}, f)

print('Saved:')
print('  models/lstm_model.pt')
print('  models/lstm_scaler.pkl')
print('\n✅  All 5 models complete!')
print('\nWhat you have now:')
print('  01_EDA          — decomposition, stationarity, ACF/PACF')
print('  02_SARIMA        — classical stats with grid search')
print('  03_Prophet       — automatic trend/seasonality/holidays')
print('  04_LightGBM      — ML with lag features')
print('  05_Hybrid        — SARIMA + XGBoost on residuals')
print('  06_LSTM          — deep learning + Monte Carlo uncertainty')
print('\nRun dashboard: streamlit run app/dashboard.py')